<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/hj_reachability_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HJ reachability basics

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from IPython.display import HTML
import matplotlib.animation as anim
import matplotlib.pyplot as plt
import functools


In [ ]:
!pip install tqdm hj-reachability

In [ ]:
import hj_reachability as hj
from hj_reachability import dynamics, sets

### Example system: `Air3D`

In [ ]:
system = hj.systems.Air3d()
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(hj.sets.Box(np.array([-6., -10., 0.]),
                                                                           np.array([20., 10., 2 * np.pi])),
                                                               (51, 40, 50),
                                                               periodic_dims=2)
values = jnp.linalg.norm(grid.states[..., :2], axis=-1) - 5

solver_settings = hj.SolverSettings.with_accuracy("very_high",
                                                  hamiltonian_postprocessor=hj.solver.backwards_reachable_tube)

### `hj.step`: propagate the HJ PDE from `(time, values)` to `target_time`.

In [ ]:
time = 0.
target_time = -2.8
target_values = hj.step(solver_settings, system, grid, time, values, target_time)

In [ ]:
angle_idx = 7
plt.jet()
plt.figure(figsize=(13, 8))
plt.contourf(grid.coordinate_vectors[0], grid.coordinate_vectors[1], target_values[:, :, angle_idx].T)
plt.colorbar()
plt.contour(grid.coordinate_vectors[0],
            grid.coordinate_vectors[1],
            target_values[:, :, angle_idx].T,
            levels=0,
            colors="black",
            linewidths=3)
print(grid.coordinate_vectors[2][angle_idx])

### Value evaluation

In [ ]:
# define a state to evaluate the value function at
# obtain a state from the grid
state = grid.states[4,5,4]

# use grid.interpolate to evaluate the value function at an interpolated state
V_value = grid.interpolate(target_values, state) # should == target_values[4,5,4]
V_value, target_values[4,5,4]


In [ ]:
state_offgridpoint = state + jnp.array([0.1, -0.2, 0.3])
grid.interpolate(target_values, state_offgridpoint)

In [ ]:
value_fn = functools.partial(grid.interpolate, values=target_values)
value_fn(state=state_offgridpoint)

### Gradient of value function evaluation

In [ ]:
# perform central differencing over target_values
dV_values =  grid.grad_values(target_values)

In [ ]:
dV_values.shape # shape is  [grid_size x state_dim]

In [ ]:
# use grid.interpolate to evaluate the gradient of value function at an interpolated state
grad_value = grid.interpolate(dV_values, state) # should == dV_values[4,5,4]
grad_value

In [ ]:
grad_value_fn = functools.partial(grid.interpolate, values=dV_values)
grad_value_fn(state = state_offgridpoint)

### Compute optimal policy

In [ ]:
state = grid.states[4,5,4] # define a state to evaluate policy at

# optimal_control_and_disturbance(self, state, time, grad_value)
a_opt, b_opt = system.optimal_control_and_disturbance(state, 0., grad_value)
a_opt, b_opt

## Double integrator system

Consider a 2D double integrator dynamics with some disturbances

$$
\begin{bmatrix} \dot{x} \\ \ddot{x} \\ \dot{y} \\ \ddot{y} \end{bmatrix}
=
\begin{bmatrix} 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 0 & 0 \end{bmatrix}
\begin{bmatrix} x \\ \dot{x} \\ y \\ \dot{y} \end{bmatrix} +
\begin{bmatrix} 0 & 0 \\ 1 & 0 \\ 0 & 0 \\ 0 & 1 \end{bmatrix}
\begin{bmatrix} u_x \\ u_y \end{bmatrix} +
\begin{bmatrix} 1 & 0 \\ 0 & 0 \\ 0 & 1 \\ 0 & 0 \end{bmatrix}
\begin{bmatrix} d_x \\ d_y \end{bmatrix}
$$

In [ ]:
class DoubleIntegrator2D(dynamics.ControlAndDisturbanceAffineDynamics):

    def __init__(self,
                 max_velocity=4.,
                 control_limit=1.,
                 disturbance_limit=0.5,
                 control_mode="min",
                 disturbance_mode="max",
                 control_space=None,
                 disturbance_space=None):
        self.max_velocity = max_velocity

        if control_space is None:
            control_space = sets.Box(jnp.array([-control_limit, -control_limit]), jnp.array([control_limit, control_limit]))
        if disturbance_space is None:
            disturbance_space = sets.Box(jnp.array([-disturbance_limit, -disturbance_limit]), jnp.array([disturbance_limit, disturbance_limit]))
        super().__init__(control_mode, disturbance_mode, control_space, disturbance_space)

    def open_loop_dynamics(self, state, time):
        x, vx, y, vy = state
        vx_clip = jnp.clip(vx, -self.max_velocity, self.max_velocity)
        vy_clip = jnp.clip(vy, -self.max_velocity, self.max_velocity)
        return jnp.array([vx_clip, 0., vy_clip, 0.])

    def control_jacobian(self, state, time):
        return jnp.array([
            [0., 0.],
            [1., 0.],
            [0., 0.],
            [0., 1.],
        ])

    def disturbance_jacobian(self, state, time):
        return jnp.array([
            [1., 0.],
            [0., 0.],
            [0., 1.],
            [0., 0.],
        ])

In [ ]:
system = DoubleIntegrator2D()
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(hj.sets.Box(np.array([-10., -5.0, -10., -5.0]),
                                                                           np.array([10., 5.0, 10., 5.0])),
                                                               (41, 31, 41, 31))
values = jnp.linalg.norm(grid.states[..., [0,2]], axis=-1) - 3

solver_settings = hj.SolverSettings.with_accuracy("very_high",
                                                  hamiltonian_postprocessor=hj.solver.backwards_reachable_tube)

In [ ]:
time = 0.
target_time = -3.0
target_values = hj.step(solver_settings, system, grid, time, values, target_time)


In [ ]:
plt.jet()
plt.figure(figsize=(7, 5))
plt.contourf(grid.coordinate_vectors[0], grid.coordinate_vectors[2], target_values[:, 15, :, 15].T)
plt.colorbar()
plt.contour(grid.coordinate_vectors[0],
            grid.coordinate_vectors[2],
            target_values[:, 15, :, 15].T,
            levels=0,
            colors="black",
            linewidths=3)
plt.axis("equal")

In [ ]:
# you can solve for multiple time steps at once
times = np.linspace(-3.0, 0., 30)
target_values = hj.solve(solver_settings, system, grid, times, values)